# 06 — Web Content Mining : Intelligence sémantique et extraction du contenu

## Objectif du notebook

Ce notebook respecte l’axe 1 demandé dans le proposal :

- conception d’un pipeline d’ingestion / prétraitement asynchrone ;
- exploitation des données non structurées, principalement les descriptions des offres ;
- extraction des compétences depuis les descriptions uniquement ;
- enrichissement des compétences existantes sans écraser la colonne `skills` ;
- démonstration Transformer / NER sur un échantillon de descriptions ;
- vectorisation TF-IDF des descriptions ;
- extraction de vecteurs de compétences ;
- classification automatique des domaines d’expertise.

## Règle importante suivie dans ce notebook

La colonne `skills` existe déjà dans le dataset principal. Elle vient du travail de nettoyage et d’harmonisation réalisé avant.

Donc :

- les offres sans description conservent leurs skills initiales ;
- les offres avec description sont analysées pour extraire des skills supplémentaires ;
- les skills extraites depuis les descriptions sont ajoutées dans une nouvelle colonne `skills_enriched` ;
- la colonne `skills` originale n’est jamais remplacée ;
- le titre du poste n’est pas utilisé pour extraire les skills.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import os
import re
import asyncio
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

warnings.filterwarnings("ignore")
tqdm.pandas()

print("Dossier courant :", os.getcwd())
print("\nFichiers CSV disponibles :")
for f in os.listdir():
    if f.endswith(".csv"):
        print("-", f)

Dossier courant : C:\Users\ROG ZEPHYRUS\Documents\web_mining\projet_final\data_final

Fichiers CSV disponibles :
- domain_classification_summary.csv
- final_data_ai_jobs_clean_2020_2026.csv
- final_data_ai_jobs_clean_2020_2026_verified.csv
- final_data_ai_jobs_clean_reduced_2020_2026.csv
- final_data_ai_jobs_content_enriched.csv
- final_data_ai_jobs_skills_exploded.csv
- final_data_ai_jobs_skills_exploded_enriched.csv
- morocco_domain_classification_summary.csv
- quality_country_distribution.csv
- quality_framework_summary.csv
- quality_missing_values_original.csv
- quality_missing_values_reduced.csv
- quality_source_distribution.csv
- quality_year_distribution.csv
- skill_vector_matrix_sample.csv
- sources_summary.csv
- tfidf_top_terms_content.csv
- tfidf_top_terms_descriptions.csv
- top_keywords_descriptions.csv


## 1. Chargement des datasets

On charge :

- le dataset principal des offres ;
- le dataset des skills explosées déjà créé à partir du dataset principal.

Le fichier `skills_exploded` est utilisé seulement pour construire un dictionnaire de compétences et analyser les skills. Il ne remplace pas le dataset principal.

In [3]:
jobs_file = "final_data_ai_jobs_clean_2020_2026.csv"
skills_file = "final_data_ai_jobs_skills_exploded.csv"

df_jobs = pd.read_csv(jobs_file)
df_skills = pd.read_csv(skills_file)

print("Dataset principal :", df_jobs.shape)
print("Dataset skills exploded :", df_skills.shape)

display(df_jobs.head(3).T)
display(df_skills.head())

Dataset principal : (751801, 30)
Dataset skills exploded : (3991435, 16)


,0,1,2
source,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset
platform,Synthetic / Global,Synthetic / Global,Synthetic / Global
job_id,1,2,3
job_title,AI Researcher,MLOps Engineer,Data Analyst
company_name,NaN,NaN,NaN
country,Canada,India,United Kingdom
city,Berlin,Tokyo,Bangalore
date_posted,2021-04-01 00:00:00,2020-04-12 00:00:00,2023-01-31 00:00:00
year,2021,2020,2023
month,4.0,4.0,1.0


,job_id,source,platform,job_title,country,city,date_posted,year,month,year_month,remote_status,experience_level,category,tools_used,original_file,skill
0,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,python
1,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,computer vision
2,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,sql
3,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,nlp
4,2,global_ai_jobs_dataset,Synthetic / Global,MLOps Engineer,India,Tokyo,2020-04-12,2020,4.0,2020-04,Remote,Entry,Recommendation Systems,Python;Spark;Docker,global_ai_jobs_dataset.csv,nlp


## 2. Vérification des colonnes nécessaires

On vérifie que les colonnes nécessaires existent :

- `job_id`
- `job_title`
- `description`
- `skills`
- `country`
- `year`
- `date_posted`

La colonne `description` est importante pour l’extraction complémentaire des compétences.

In [4]:
required_cols = ["job_id", "job_title", "description", "skills", "country", "year", "date_posted"]

for col in required_cols:
    if col in df_jobs.columns:
        print("Existe :", col)
    else:
        print("Manquante :", col)

print("\nNombre total d'offres :", len(df_jobs))
print("Offres avec skills :", df_jobs["skills"].notna().sum())
print("Offres sans skills :", df_jobs["skills"].isna().sum())
print("Offres avec description :", df_jobs["description"].notna().sum())
print("Offres sans description :", df_jobs["description"].isna().sum())

Existe : job_id
Existe : job_title
Existe : description
Existe : skills
Existe : country
Existe : year
Existe : date_posted

Nombre total d'offres : 751801
Offres avec skills : 751476
Offres sans skills : 325
Offres avec description : 3061
Offres sans description : 748740


## 3. Fonctions de nettoyage du texte

Ces fonctions servent à nettoyer uniquement les descriptions.

Elles ne modifient pas la colonne `skills`.

Le nettoyage consiste à :

- convertir en texte sûr ;
- mettre en minuscules ;
- supprimer les liens ;
- supprimer les caractères inutiles ;
- réduire les espaces multiples.

In [5]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)

def clean_content_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9+#.\s/-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 4. Pipeline asynchrone de prétraitement des descriptions

Le proposal demande un pipeline d’ingestion asynchrone.

Ici, le pipeline traite uniquement les offres ayant une description exploitable.  
Les offres sans description ne sont pas traitées par l’extraction de skills depuis description.

Le résultat est la colonne :

`description_text_clean`

In [6]:
# Préparation des descriptions
df_jobs["description_text"] = df_jobs["description"].apply(safe_text)

description_mask = df_jobs["description_text"].str.strip().str.len() > 20
df_desc = df_jobs.loc[description_mask, ["job_id", "description_text"]].copy().reset_index()

print("Nombre d'offres avec description exploitable :", df_desc.shape[0])

async def process_description_batch(batch_df):
    await asyncio.sleep(0)
    batch_df = batch_df.copy()
    batch_df["description_text_clean"] = batch_df["description_text"].apply(clean_content_text)
    return batch_df

async def async_description_pipeline(df_input, batch_size=500):
    tasks = []
    for start in range(0, len(df_input), batch_size):
        batch = df_input.iloc[start:start + batch_size]
        tasks.append(process_description_batch(batch))

    processed_batches = []
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Prétraitement asynchrone descriptions"):
        processed_batches.append(await task)

    if len(processed_batches) == 0:
        return pd.DataFrame()

    return pd.concat(processed_batches, ignore_index=True)

df_desc_processed = await async_description_pipeline(df_desc, batch_size=500)

# Initialiser la colonne pour toutes les offres
df_jobs["description_text_clean"] = ""

# Réinjecter uniquement les descriptions nettoyées dans le dataset principal
if not df_desc_processed.empty:
    df_jobs.loc[df_desc_processed["index"], "description_text_clean"] = df_desc_processed["description_text_clean"].values

print("Prétraitement terminé.")
display(df_jobs.loc[description_mask, ["job_title", "description_text_clean"]].head())

Nombre d'offres avec description exploitable : 3061


Prétraitement asynchrone descriptions: 100%|█████████████████████████████████████████████| 7/7 [00:00<00:00, 11.24it/s]

Prétraitement terminé.


,job_title,description_text_clean
120000,Senior Full stack React Developer,are you a talented senior developer looking fo...
120001,JAVA JEE Developer (M/F),offre d emploi maroc java jee developer m/f - ...
120002,Engineer Software,at verint we believe customer engagement is th...
120003,"Senior Machine Learning Engineer, Personalization",headquarters sweden url you ll join a team wor...
120004,DevOps Engineer,headquarters remote latin america on this jour...


## 5. Construction du dictionnaire de compétences

Le dictionnaire de compétences est construit à partir du fichier `skills_exploded`, lui-même créé depuis le dataset principal.

Il est complété par une liste manuelle de compétences importantes en Data / IA.

Ce dictionnaire sera utilisé pour extraire des compétences uniquement depuis les descriptions.

In [7]:
top_skill_list = (
    df_skills["skill"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .value_counts()
    .head(300)
    .index
    .tolist()
)

manual_skills = [
    "python", "sql", "r", "java", "scala", "spark", "hadoop", "kafka", "airflow",
    "aws", "azure", "gcp", "docker", "kubernetes", "mlflow", "databricks",
    "machine learning", "deep learning", "nlp", "llm", "generative ai",
    "tensorflow", "pytorch", "scikit-learn", "computer vision",
    "power bi", "tableau", "excel", "snowflake", "dbt", "nosql", "mongodb",
    "pandas", "numpy", "matplotlib", "seaborn", "fastapi", "flask",
    "pyspark", "bigquery", "redshift", "oracle", "sql server"
]

skill_dictionary = sorted(set(top_skill_list + manual_skills), key=len, reverse=True)

print("Nombre de skills dans le dictionnaire :", len(skill_dictionary))
print(skill_dictionary[:40])

Nombre de skills dans le dictionnaire : 301
['natural language processing', 'artificial intelligence', 'reinforcement learning', 'problemsolving skills', 'business intelligence', 'organizational skills', 'communication skills', 'statistical modeling', 'software engineering', 'software development', 'statistical analysis', 'interpersonal skills', 'relational databases', 'presentation skills', 'attention to detail', 'data transformation', 'data interpretation', 'predictive modeling', 'project management', 'data visualization', 'hypothesis testing', 'analytical skills', 'critical thinking', 'business analysis', 'agile development', 'bachelor s degree', 'data manipulation', 'data architecture', 'data engineering', 'data warehousing', 'computer science', 'data integration', 'machine learning', 'data extraction', 'shell scripting', 'troubleshooting', 'time management', 'data collection', 'microsoft teams', 'computer vision']


## 6. Extraction des compétences depuis les descriptions uniquement

Cette étape est très importante.

Contrairement à une extraction sur `job_title + description + skills`, ici on applique l’extraction uniquement sur :

`description_text_clean`

Donc :

- si l’offre n’a pas de description, aucune extraction complémentaire n’est faite ;
- si l’offre a une description, on cherche les compétences mentionnées dans cette description ;
- la colonne `skills` originale n’est pas modifiée.

Colonnes créées :

- `skills_extracted_from_description`
- `num_skills_extracted_description`

In [8]:
def extract_skills_dictionary(text, skill_dict):
    text = clean_content_text(text)
    found = []

    if len(text) <= 20:
        return []

    for skill in skill_dict:
        pattern = r"(?<![a-zA-Z0-9+#.])" + re.escape(skill) + r"(?![a-zA-Z0-9+#.])"
        if re.search(pattern, text):
            found.append(skill)

    return list(dict.fromkeys(found))

# Initialiser avec des listes vides
df_jobs["skills_extracted_from_description"] = [[] for _ in range(len(df_jobs))]

# Appliquer uniquement aux offres avec description exploitable
mask_desc_clean = df_jobs["description_text_clean"].str.len() > 20

df_jobs.loc[mask_desc_clean, "skills_extracted_from_description"] = (
    df_jobs.loc[mask_desc_clean, "description_text_clean"]
    .progress_apply(lambda x: extract_skills_dictionary(x, skill_dictionary))
)

df_jobs["num_skills_extracted_description"] = df_jobs["skills_extracted_from_description"].apply(len)

print("Offres avec description exploitable :", mask_desc_clean.sum())
print("Offres avec au moins une skill extraite depuis description :",
      (df_jobs["num_skills_extracted_description"] > 0).sum())

display(df_jobs.loc[mask_desc_clean, [
    "job_title",
    "description_text_clean",
    "skills",
    "skills_extracted_from_description",
    "num_skills_extracted_description"
]].head(10))

100%|██████████████████████████████████████████████████████████████████████████████| 3061/3061 [00:54<00:00, 56.59it/s]


Offres avec description exploitable : 3061
Offres avec au moins une skill extraite depuis description : 3047


,job_title,description_text_clean,skills,skills_extracted_from_description,num_skills_extracted_description
120000,Senior Full stack React Developer,are you a talented senior developer looking fo...,".net, ai, angular, aws, azure, c#, c++, data a...","[organizational skills, software development, ...",41
120001,JAVA JEE Developer (M/F),offre d emploi maroc java jee developer m/f - ...,"ai, angular, aws, azure, big data, cassandra, ...","[software engineering, software development, r...",37
120002,Engineer Software,at verint we believe customer engagement is th...,".net, ai, aws, azure, c#, ci, cd, cloud, docke...","[software engineering, software development, r...",45
120003,"Senior Machine Learning Engineer, Personalization",headquarters sweden url you ll join a team wor...,"ai, apache spark, artificial intelligence, aws...","[natural language processing, artificial intel...",16
120004,DevOps Engineer,headquarters remote latin america on this jour...,"ai, aws, bash, cloud, gcp, generative ai, gola...","[machine learning, generative ai, documentatio...",17
120005,AWS Solutions Architect (M/F),offre d emploi maroc aws solutions architect m...,"ai, aws, big data, cloud, data analytics, dock...","[data management, data processing, cloud compu...",24
120006,DevOps Engineer (M/F),offre d emploi maroc devops engineer m/f - / t...,"ai, angular, ansible, aws, azure, ci, cd, clou...","[communication skills, software engineering, t...",36
120007,Senior DevOps Engineer,headquarters calgary who we are at zayzoon we ...,"ansible, aws, bash, ci, cd, cloud, cybersecuri...","[engineering, postgresql, automation, monitori...",21
120008,DATASTAGE Architect (M/F),offre d emploi maroc datastage architect m/f -...,"ai, aws, azure, cloud, google cloud, informati...","[interpersonal skills, attention to detail, tr...",22
120009,Senior DevOps Engineer,headquarters about kasha kasha will disrupt th...,"aws, azure, ci, cd, cloud, gcp, google cloud, ...","[software engineering, computer science, troub...",26


## 7. Fusion des skills existantes avec les skills extraites depuis les descriptions

La colonne `skills` contient déjà les compétences finales issues du nettoyage.

On conserve cette colonne comme référence principale.

Ensuite, on crée une nouvelle colonne :

`skills_enriched`

Cette colonne contient :

`skills existantes + skills extraites depuis description`

Les doublons sont supprimés.

Les offres sans description gardent simplement leurs skills initiales.

In [9]:
def split_existing_skills(skills):
    if pd.isna(skills):
        return []

    return [
        s.strip().lower()
        for s in str(skills).split(",")
        if s.strip() and s.strip().lower() not in ["nan", "none", "[]"]
    ]

def merge_skills(existing_skills, extracted_skills):
    existing = split_existing_skills(existing_skills)

    if isinstance(extracted_skills, list):
        extracted = [
            str(s).strip().lower()
            for s in extracted_skills
            if str(s).strip()
        ]
    else:
        extracted = []

    merged = sorted(set(existing + extracted))

    if len(merged) == 0:
        return np.nan

    return ", ".join(merged)

def get_new_skills(existing_skills, extracted_skills):
    existing = set(split_existing_skills(existing_skills))

    if isinstance(extracted_skills, list):
        extracted = set([
            str(s).strip().lower()
            for s in extracted_skills
            if str(s).strip()
        ])
    else:
        extracted = set()

    return sorted(extracted - existing)

df_jobs["skills_enriched"] = df_jobs.apply(
    lambda row: merge_skills(
        row["skills"],
        row["skills_extracted_from_description"]
    ),
    axis=1
)

df_jobs["new_skills_from_description"] = df_jobs.apply(
    lambda row: get_new_skills(
        row["skills"],
        row["skills_extracted_from_description"]
    ),
    axis=1
)

df_jobs["num_skills_original"] = df_jobs["skills"].apply(lambda x: len(split_existing_skills(x)))
df_jobs["num_skills_enriched"] = df_jobs["skills_enriched"].apply(lambda x: len(split_existing_skills(x)))
df_jobs["num_new_skills_from_description"] = df_jobs["new_skills_from_description"].apply(len)

print("Offres avec skills originales :", df_jobs["skills"].notna().sum())
print("Offres avec description exploitable :", mask_desc_clean.sum())
print("Offres enrichies avec au moins une nouvelle skill :",
      (df_jobs["num_new_skills_from_description"] > 0).sum())
print("Moyenne skills originales :", round(df_jobs["num_skills_original"].mean(), 2))
print("Moyenne skills enrichies :", round(df_jobs["num_skills_enriched"].mean(), 2))

display(df_jobs.loc[mask_desc_clean, [
    "job_title",
    "skills",
    "skills_extracted_from_description",
    "new_skills_from_description",
    "skills_enriched",
    "num_skills_original",
    "num_skills_enriched",
    "num_new_skills_from_description"
]].head(20))

Offres avec skills originales : 751476
Offres avec description exploitable : 3061
Offres enrichies avec au moins une nouvelle skill : 2927
Moyenne skills originales : 5.31
Moyenne skills enrichies : 5.35


,job_title,skills,skills_extracted_from_description,new_skills_from_description,skills_enriched,num_skills_original,num_skills_enriched,num_new_skills_from_description
120000,Senior Full stack React Developer,".net, ai, angular, aws, azure, c#, c++, data a...","[organizational skills, software development, ...","[a, communication, data engineering, data scie...",".net, a, ai, angular, aws, azure, c#, c++, com...",23,42,19
120001,JAVA JEE Developer (M/F),"ai, angular, aws, azure, big data, cassandra, ...","[software engineering, software development, r...","[a, agile, aurora, c, communication, couchbase...","a, agile, ai, angular, aurora, aws, azure, big...",20,38,18
120002,Engineer Software,".net, ai, aws, azure, c#, ci, cd, cloud, docke...","[software engineering, software development, r...","[a, agile, api, bachelor s degree, collaborati...",".net, a, agile, ai, api, aws, azure, bachelor ...",26,50,24
120003,"Senior Machine Learning Engineer, Personalization","ai, apache spark, artificial intelligence, aws...","[natural language processing, artificial intel...","[a, data processing, data science, engineering...","a, ai, apache spark, artificial intelligence, ...",11,17,6
120004,DevOps Engineer,"ai, aws, bash, cloud, gcp, generative ai, gola...","[machine learning, generative ai, documentatio...","[a, aurora, devops, documentation]","a, ai, aurora, aws, bash, cloud, devops, docum...",15,19,4
120005,AWS Solutions Architect (M/F),"ai, aws, big data, cloud, data analytics, dock...","[data management, data processing, cloud compu...","[a, analytics, api, c, centos, cloud computing...","a, ai, analytics, api, aws, big data, c, cento...",19,29,10
120006,DevOps Engineer (M/F),"ai, angular, ansible, aws, azure, ci, cd, clou...","[communication skills, software engineering, t...","[a, agile, c, communication, communication ski...","a, agile, ai, angular, ansible, aws, azure, c,...",22,38,16
120007,Senior DevOps Engineer,"ansible, aws, bash, ci, cd, cloud, cybersecuri...","[engineering, postgresql, automation, monitori...","[a, automation, chef, databases, devops, engin...","a, ansible, automation, aws, bash, cd, chef, c...",17,26,9
120008,DATASTAGE Architect (M/F),"ai, aws, azure, cloud, google cloud, informati...","[interpersonal skills, attention to detail, tr...","[a, attention to detail, c, cloud computing, d...","a, ai, attention to detail, aws, azure, c, clo...",17,27,10
120009,Senior DevOps Engineer,"aws, azure, ci, cd, cloud, gcp, google cloud, ...","[software engineering, computer science, troub...","[a, automation, computer science, databases, d...","a, automation, aws, azure, cd, ci, cloud, comp...",13,28,15


## 8. Démonstration Transformer / NER sur un échantillon de descriptions

Le proposal demande le déploiement de modèles Transformers / NER.

Cette cellule applique un modèle NER orienté compétences sur un petit échantillon de descriptions.

Cette étape est une démonstration NLP avancée.  
Elle ne remplace pas la colonne `skills` et ne remplace pas `skills_enriched`.

Si le modèle ne peut pas être téléchargé ou exécuté, le notebook continue avec les méthodes précédentes.

In [10]:
USE_TRANSFORMER_NER = True
TRANSFORMER_SAMPLE_SIZE = 30

SKILL_NER_MODEL_NAME = "Nucha/Nucha_SkillNER_BERT"

transformer_ner_results = []

if USE_TRANSFORMER_NER:
    try:
        from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

        print("Chargement du modèle Transformer NER :", SKILL_NER_MODEL_NAME)
        tokenizer = AutoTokenizer.from_pretrained(SKILL_NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(SKILL_NER_MODEL_NAME)

        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple"
        )

        df_ner_sample = df_jobs.loc[
            df_jobs["description_text_clean"].str.len() > 20,
            ["job_id", "job_title", "description_text_clean"]
        ].head(TRANSFORMER_SAMPLE_SIZE).copy()

        for idx, row in tqdm(df_ner_sample.iterrows(), total=len(df_ner_sample), desc="NER Transformer"):
            text = row["description_text_clean"][:1500]
            entities = ner_pipeline(text)

            for ent in entities:
                transformer_ner_results.append({
                    "job_id": row.get("job_id", idx),
                    "job_title": row.get("job_title", None),
                    "entity": ent.get("word", ""),
                    "entity_group": ent.get("entity_group", ""),
                    "score": ent.get("score", None)
                })

        df_transformer_ner = pd.DataFrame(transformer_ner_results)
        print("Nombre d'entités extraites par Transformer :", df_transformer_ner.shape[0])
        display(df_transformer_ner.head(20))

        df_transformer_ner.to_csv(
            "transformer_ner_skills_sample.csv",
            index=False,
            encoding="utf-8-sig"
        )

    except Exception as e:
        print("Le modèle Transformer NER n'a pas pu être exécuté.")
        print("Raison :", e)
        print("Le notebook continue avec l'extraction par dictionnaire depuis les descriptions.")
        df_transformer_ner = pd.DataFrame()
else:
    print("Transformer NER désactivé.")
    df_transformer_ner = pd.DataFrame()

Chargement du modèle Transformer NER : Nucha/Nucha_SkillNER_BERT
Le modèle Transformer NER n'a pas pu être exécuté.
Raison : Can't load config for 'Nucha/Nucha_SkillNER_BERT'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Nucha/Nucha_SkillNER_BERT' is the correct path to a directory containing a config.json file
Le notebook continue avec l'extraction par dictionnaire depuis les descriptions.


## 9. Vectorisation TF-IDF des descriptions

Cette étape transforme les descriptions nettoyées en vecteurs numériques.

On utilise uniquement les descriptions disponibles, car ce sont les données non structurées textuelles.

La matrice TF-IDF permet d’identifier les termes les plus importants dans les descriptions.

In [11]:
df_desc_for_tfidf = df_jobs.loc[
    df_jobs["description_text_clean"].str.len() > 20,
    ["job_id", "description_text_clean"]
].copy()

print("Nombre de descriptions utilisées pour TF-IDF :", df_desc_for_tfidf.shape[0])

if df_desc_for_tfidf.shape[0] > 0:
    tfidf_vectorizer = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        stop_words="english",
        min_df=3
    )

    tfidf_matrix = tfidf_vectorizer.fit_transform(df_desc_for_tfidf["description_text_clean"])

    print("Shape de la matrice TF-IDF :", tfidf_matrix.shape)

    tfidf_terms = pd.DataFrame({
        "term": tfidf_vectorizer.get_feature_names_out(),
        "global_weight": np.asarray(tfidf_matrix.sum(axis=0)).ravel()
    }).sort_values("global_weight", ascending=False)

    display(tfidf_terms.head(30))

    tfidf_terms.to_csv(
        "tfidf_top_terms_descriptions.csv",
        index=False,
        encoding="utf-8-sig"
    )
else:
    print("Aucune description disponible pour TF-IDF.")
    tfidf_terms = pd.DataFrame()

Nombre de descriptions utilisées pour TF-IDF : 3061
Shape de la matrice TF-IDF : (3061, 2000)


,term,global_weight
446,data,421.361041
734,experience,212.331421
78,ai,196.400019
1963,work,124.661951
237,business,123.229862
1062,learning,111.648984
1805,team,94.731688
1689,skills,92.573468
1116,machine,88.648737
1117,machine learning,87.586194


## 10. Vectorisation des compétences enrichies

Cette étape crée des vecteurs de compétences à partir de `skills_enriched`.

Chaque offre devient un vecteur binaire :

- 1 si la compétence est présente ;
- 0 sinon.

Cela répond à la notion de vecteurs de compétences demandée dans le proposal.

Pour éviter un fichier trop lourd, on sauvegarde seulement un échantillon de la matrice.

In [12]:
df_skill_vectors = df_jobs.copy()

df_skill_vectors["skills_vector_list"] = df_skill_vectors["skills_enriched"].apply(split_existing_skills)

top_100_skills = (
    pd.Series([skill for skills in df_skill_vectors["skills_vector_list"] for skill in skills])
    .value_counts()
    .head(100)
    .index
    .tolist()
)

df_skill_vectors["skills_vector_list_top"] = df_skill_vectors["skills_vector_list"].apply(
    lambda skills: [s for s in skills if s in top_100_skills]
)

mlb = MultiLabelBinarizer(classes=top_100_skills)
skill_matrix = mlb.fit_transform(df_skill_vectors["skills_vector_list_top"])

df_skill_matrix = pd.DataFrame(skill_matrix, columns=mlb.classes_)

print("Shape de la matrice de compétences :", df_skill_matrix.shape)
display(df_skill_matrix.head())

df_skill_matrix.head(5000).to_csv(
    "skill_vector_matrix_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé : skill_vector_matrix_sample.csv")

Shape de la matrice de compétences : (751801, 100)


,sql,python,aws,r,tableau,excel,azure,spark,power bi,tensorflow,...,bitbucket,db2,splunk,mongo,redis,flask,plotly,perl,css,seaborn
0,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Fichier sauvegardé : skill_vector_matrix_sample.csv


## 11. Embeddings Transformer des descriptions

Cette étape génère des embeddings sémantiques avec un modèle Sentence Transformer.

Les embeddings sont calculés uniquement sur un échantillon de descriptions pour éviter un temps d’exécution trop long.

Ils permettent de représenter le sens du texte sous forme de vecteurs.

In [13]:
USE_SENTENCE_TRANSFORMER = True
EMBEDDING_SAMPLE_SIZE = 1000

if USE_SENTENCE_TRANSFORMER:
    try:
        from sentence_transformers import SentenceTransformer

        embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
        print("Chargement du modèle d'embeddings :", embedding_model_name)

        embedder = SentenceTransformer(embedding_model_name)

        texts_for_embeddings = (
            df_jobs.loc[df_jobs["description_text_clean"].str.len() > 20, "description_text_clean"]
            .head(EMBEDDING_SAMPLE_SIZE)
            .tolist()
        )

        embeddings = embedder.encode(
            texts_for_embeddings,
            show_progress_bar=True,
            batch_size=32
        )

        print("Shape embeddings :", embeddings.shape)

        np.save("description_embeddings_sample.npy", embeddings)
        print("Embeddings sauvegardés : description_embeddings_sample.npy")

    except Exception as e:
        print("Embeddings Transformer non exécutés.")
        print("Raison :", e)
else:
    print("Sentence Transformer désactivé.")

Chargement du modèle d'embeddings : sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 32/32 [00:04<00:00,  7.36it/s]

Shape embeddings : (1000, 384)
Embeddings sauvegardés : description_embeddings_sample.npy


## 12. Classification automatique des domaines d’expertise

Cette étape classe automatiquement les offres en familles de métiers :

- Data Engineering
- Data Analysis / BI
- Machine Learning / AI
- Data Science
- MLOps / Cloud Data
- Database / Data Warehouse
- Other Data / AI

La classification utilise le titre, la description disponible et les skills enrichies.  
Le titre est utilisé uniquement pour la classification du domaine, pas pour l’extraction des compétences.

In [14]:
def classify_expertise_domain(text):
    text = clean_content_text(text)

    if any(k in text for k in ["mlops", "kubernetes", "docker", "mlflow", "ci/cd", "devops", "cloud engineer"]):
        return "MLOps / Cloud Data"

    if any(k in text for k in [
        "machine learning", "deep learning", "nlp", "llm", "generative ai",
        "computer vision", "tensorflow", "pytorch", "scikit-learn", "ai engineer",
        "artificial intelligence"
    ]):
        return "Machine Learning / AI"

    if any(k in text for k in [
        "data engineer", "etl", "spark", "hadoop", "kafka", "airflow",
        "databricks", "data pipeline", "data warehouse"
    ]):
        return "Data Engineering"

    if any(k in text for k in ["data scientist", "data science", "statistics", "statistical modeling"]):
        return "Data Science"

    if any(k in text for k in [
        "data analyst", "business analyst", "business intelligence",
        "power bi", "tableau", "excel", "dashboard", "reporting"
    ]):
        return "Data Analysis / BI"

    if any(k in text for k in ["database administrator", "dba", "sql server", "oracle", "mongodb", "database"]):
        return "Database / Data Warehouse"

    return "Other Data / AI"

df_jobs["content_text_for_classification"] = (
    df_jobs["job_title"].apply(safe_text) + " " +
    df_jobs["description_text_clean"].apply(safe_text) + " " +
    df_jobs["skills_enriched"].apply(safe_text)
)

df_jobs["expertise_domain"] = df_jobs["content_text_for_classification"].apply(classify_expertise_domain)

domain_summary = (
    df_jobs["expertise_domain"]
    .value_counts()
    .reset_index()
)

domain_summary.columns = ["expertise_domain", "number_of_jobs"]
domain_summary["percentage"] = round(domain_summary["number_of_jobs"] / len(df_jobs) * 100, 2)

display(domain_summary)

domain_summary.to_csv(
    "domain_classification_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé : domain_classification_summary.csv")

,expertise_domain,number_of_jobs,percentage
0,Data Engineering,219296,29.17
1,Data Analysis / BI,193983,25.80
2,Machine Learning / AI,153308,20.39
3,Data Science,87358,11.62
4,MLOps / Cloud Data,59802,7.95
5,Other Data / AI,32473,4.32
6,Database / Data Warehouse,5581,0.74


Fichier sauvegardé : domain_classification_summary.csv


## 13. Focus Maroc : domaines d’expertise

Cette partie analyse la répartition des domaines d’expertise pour les offres localisées au Maroc.

In [15]:
df_morocco = df_jobs[df_jobs["country"] == "Morocco"].copy()

print("Nombre d'offres Maroc :", df_morocco.shape[0])

morocco_domain_summary = (
    df_morocco["expertise_domain"]
    .value_counts()
    .reset_index()
)

morocco_domain_summary.columns = ["expertise_domain", "number_of_jobs"]
morocco_domain_summary["percentage"] = round(
    morocco_domain_summary["number_of_jobs"] / max(len(df_morocco), 1) * 100,
    2
)

display(morocco_domain_summary)

morocco_domain_summary.to_csv(
    "morocco_domain_classification_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé : morocco_domain_classification_summary.csv")

Nombre d'offres Maroc : 1058


,expertise_domain,number_of_jobs,percentage
0,Data Engineering,361,34.12
1,Data Analysis / BI,299,28.26
2,MLOps / Cloud Data,148,13.99
3,Machine Learning / AI,102,9.64
4,Data Science,87,8.22
5,Other Data / AI,58,5.48
6,Database / Data Warehouse,3,0.28


Fichier sauvegardé : morocco_domain_classification_summary.csv


## 14. Mots-clés fréquents dans les descriptions

Cette analyse utilise uniquement les descriptions disponibles.

Elle permet d’identifier les termes dominants dans les descriptions d’offres.

In [16]:
df_desc_keywords = df_jobs[df_jobs["description_text_clean"].str.len() > 20].copy()

print("Nombre d'offres avec description exploitable :", df_desc_keywords.shape[0])

if df_desc_keywords.shape[0] > 0:
    desc_sample = df_desc_keywords["description_text_clean"].sample(
        min(50000, len(df_desc_keywords)),
        random_state=42
    )

    count_vectorizer = CountVectorizer(
        stop_words="english",
        max_features=100,
        ngram_range=(1, 2),
        min_df=5
    )

    X_count = count_vectorizer.fit_transform(desc_sample)

    keywords_df = pd.DataFrame({
        "keyword": count_vectorizer.get_feature_names_out(),
        "count": np.asarray(X_count.sum(axis=0)).ravel()
    }).sort_values("count", ascending=False)

    display(keywords_df.head(30))

    keywords_df.to_csv(
        "top_keywords_descriptions.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Fichier sauvegardé : top_keywords_descriptions.csv")
else:
    print("Pas de descriptions disponibles.")
    keywords_df = pd.DataFrame()

Nombre d'offres avec description exploitable : 3061


,keyword,count
20,data,25741
32,experience,12561
1,ai,7718
95,work,7164
12,business,5887
86,team,4876
42,learning,4825
78,skills,4337
91,time,4026
29,engineering,3990


Fichier sauvegardé : top_keywords_descriptions.csv


## 15. Création du nouveau dataset skills exploded enrichi

Le fichier `final_data_ai_jobs_skills_exploded.csv` initial a été créé à partir de la colonne `skills`.

Après enrichissement depuis les descriptions, on crée une nouvelle version :

`final_data_ai_jobs_skills_exploded_enriched.csv`

Elle est créée à partir de `skills_enriched`.

On ne supprime pas l’ancien fichier.

In [17]:
df_for_skills = df_jobs.copy()

df_for_skills = df_for_skills[df_for_skills["skills_enriched"].notna()].copy()
df_for_skills["skill"] = df_for_skills["skills_enriched"].str.split(",")

df_for_skills = df_for_skills.explode("skill")

df_for_skills["skill"] = (
    df_for_skills["skill"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df_for_skills = df_for_skills[
    (df_for_skills["skill"] != "") &
    (df_for_skills["skill"] != "nan") &
    (df_for_skills["skill"] != "none") &
    (df_for_skills["skill"] != "[]")
].copy()

skills_columns = [
    "job_id",
    "job_title",
    "country",
    "city",
    "date_posted",
    "year",
    "month",
    "year_month",
    "remote_status",
    "category",
    "platform",
    "source",
    "original_file",
    "skill"
]

available_skill_columns = [col for col in skills_columns if col in df_for_skills.columns]
df_skills_enriched = df_for_skills[available_skill_columns].copy()

print("Shape skills enriched exploded :", df_skills_enriched.shape)
print("Nombre de skills différents :", df_skills_enriched["skill"].nunique())

display(df_skills_enriched.head())

df_skills_enriched.to_csv(
    "final_data_ai_jobs_skills_exploded_enriched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé : final_data_ai_jobs_skills_exploded_enriched.csv")

Shape skills enriched exploded : (4019147, 14)
Nombre de skills différents : 28277


,job_id,job_title,country,city,date_posted,year,month,year_month,remote_status,category,platform,source,original_file,skill
0,1,AI Researcher,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,computer vision
0,1,AI Researcher,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,nlp
0,1,AI Researcher,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,python
0,1,AI Researcher,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,sql
1,2,MLOps Engineer,India,Tokyo,2020-04-12 00:00:00,2020,4.0,2020-04,Remote,Recommendation Systems,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,computer vision


Fichier sauvegardé : final_data_ai_jobs_skills_exploded_enriched.csv


## 16. Sauvegarde du dataset Web Content Mining enrichi

On sauvegarde un dataset enrichi contenant :

- les skills originales ;
- les skills extraites depuis les descriptions ;
- les nouvelles skills ajoutées ;
- les skills enrichies ;
- le domaine d’expertise.

La colonne `skills` originale est conservée.

In [18]:
content_columns = [
    "job_id",
    "job_title",
    "country",
    "city",
    "date_posted",
    "year",
    "month",
    "year_month",
    "remote_status",
    "skills",
    "skills_extracted_from_description",
    "new_skills_from_description",
    "skills_enriched",
    "num_skills_original",
    "num_skills_enriched",
    "num_new_skills_from_description",
    "expertise_domain",
    "source",
    "platform",
    "original_file"
]

available_content_columns = [col for col in content_columns if col in df_jobs.columns]

df_content_enriched = df_jobs[available_content_columns].copy()

df_content_enriched.to_csv(
    "final_data_ai_jobs_content_enriched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé : final_data_ai_jobs_content_enriched.csv")
print("Shape :", df_content_enriched.shape)

display(df_content_enriched.tail())

Fichier sauvegardé : final_data_ai_jobs_content_enriched.csv
Shape : (751801, 20)


,job_id,job_title,country,city,date_posted,year,month,year_month,remote_status,skills,skills_extracted_from_description,new_skills_from_description,skills_enriched,num_skills_original,num_skills_enriched,num_new_skills_from_description,expertise_domain,source,platform,original_file
751796,https://aijobs.net/job/senior-devops-consultan...,Senior DevOps Consultant,India,"Bengaluru, Karnataka, IN",2026-05-06 05:49:35,2026,5.0,2026-05,Remote,"aks, application insights, azure, azure key va...","[business intelligence, software development, ...","[a, agile, ai, airflow, analytics, apache spar...","a, agile, ai, airflow, aks, analytics, apache ...",21,71,50,MLOps / Cloud Data,aijobs_raw,aijobs.net,aijobs_raw.csv
751797,https://aijobs.net/job/senior-embedded-integra...,Senior Embedded Integration Engineer,India,"Chennai, Tamil Nadu, IN",2026-05-06 09:48:06,2026,5.0,2026-05,Remote,"api, automated testing, can, ci, cd, cause ana...","[business intelligence, software development, ...","[a, agile, ai, airflow, analytics, apache spar...","a, agile, ai, airflow, analytics, apache spark...",21,73,52,MLOps / Cloud Data,aijobs_raw,aijobs.net,aijobs_raw.csv
751798,https://aijobs.net/job/statistician-solid-dosa...,Statistician - Solid Dosage Forms,United,"Durham, North Carolina, US",2026-05-06 09:48:27,2026,5.0,2026-05,On-site,"acceptance criteria, analytical method develop...","[data management, data governance, data report...","[a, ai, analytics, aws, azure, clustering, con...","a, acceptance criteria, ai, analytical method ...",12,29,17,MLOps / Cloud Data,aijobs_raw,aijobs.net,aijobs_raw.csv
751799,https://aijobs.net/job/storage-infrastructure-...,Storage / Infrastructure Specialist (all gender),Germany,"Nürnberg, Bavaria, DE",2026-05-06 07:49:23,2026,5.0,2026-05,On-site,"change management, fibre channel, itil, incide...","[business intelligence, engineering, monitorin...","[a, ai, analytics, business intelligence, engi...","a, ai, analytics, business intelligence, chang...",12,18,6,Data Analysis / BI,aijobs_raw,aijobs.net,aijobs_raw.csv
751800,https://aijobs.net/job/werkstudent-mwd-im-bere...,Werkstudent (m/w/d) im Bereich Embedded Develo...,Germany,"Dortmund, North Rhine-Westphalia, DE",2026-05-06 11:47:59,2026,5.0,2026-05,Remote,"c#, c++, embedded linux, grpc, git, petalinux,...","[software development, data transformation, da...","[a, agile, ai, analytics, aws, azure, bash, c,...","a, agile, ai, analytics, aws, azure, bash, c, ...",10,45,35,MLOps / Cloud Data,aijobs_raw,aijobs.net,aijobs_raw.csv


## 17. Synthèse finale

Ce notebook a réalisé l’axe 1 Web Content Mining en respectant la logique suivante :

- les skills existantes ont été conservées ;
- l’extraction automatique a été appliquée uniquement aux descriptions disponibles ;
- les skills extraites depuis les descriptions ont été ajoutées dans `skills_enriched` ;
- les offres sans description gardent leurs skills initiales ;
- une démonstration Transformer / NER a été incluse ;
- des vecteurs TF-IDF et des vecteurs de compétences ont été générés ;
- les offres ont été classées automatiquement par domaine d’expertise ;
- un nouveau fichier skills exploded enrichi a été généré à partir du dataset principal enrichi.

In [19]:
# Vérifier que les deux colonnes existent
print("Colonnes avant renommage :")
print([col for col in df_jobs.columns if "skills" in col.lower()])

# Renommer avec une colonne temporaire
df_jobs = df_jobs.rename(columns={
    "skills": "skills_original_temp",
    "skills_enriched": "skills"
})

df_jobs = df_jobs.rename(columns={
    "skills_original_temp": "skills_enriched"
})

# Vérifier le résultat
print("\nColonnes après renommage :")
print([col for col in df_jobs.columns if "skills" in col.lower()])

# Afficher les données
display(df_jobs[
    [
        "job_title",
        "description",
        "skills",
        "skills_enriched",
        "skills_extracted_from_description",
        "new_skills_from_description"
    ]
].tail(20))

Colonnes avant renommage :
['skills', 'skills_clean', 'skills_original', 'skills_extracted_from_description', 'num_skills_extracted_description', 'skills_enriched', 'new_skills_from_description', 'num_skills_original', 'num_skills_enriched', 'num_new_skills_from_description']

Colonnes après renommage :
['skills_enriched', 'skills_clean', 'skills_original', 'skills_extracted_from_description', 'num_skills_extracted_description', 'skills', 'new_skills_from_description', 'num_skills_original', 'num_skills_enriched', 'num_new_skills_from_description']


,job_title,description,skills,skills_enriched,skills_extracted_from_description,new_skills_from_description
751781,Applied Computer Vision Engineer - Data Driven...,Applied Computer Vision Engineer - Data Driven...,"a, ai, airflow, analytics, apache spark, api, ...","azure blob, azure blob storage, azure ml, azur...","[business intelligence, software development, ...","[a, ai, airflow, analytics, apache spark, api,..."
751782,Computational Biologist - Spatial Multi-Omics,Computational Biologist - Spatial Multi-Omics ...,"a, ai, airflow, analytics, api, artificial int...","batch correction, data visualization, deep lea...","[artificial intelligence, data transformation,...","[a, ai, airflow, analytics, api, artificial in..."
751783,Frontier AI Research Lead,Frontier AI Research Lead - Georgetown Univers...,"a, agile, ai, analytics, artificial intelligen...","artificial intelligence, data analysis, data v...","[artificial intelligence, reinforcement learni...","[a, agile, ai, analytics, c#, c++, cloud, clou..."
751784,Machine Learning Engineer,"Machine Learning Engineer - Vadodara, GJ, Indi...","a, agile, ai, airflow, analytics, apache spark...","ci, cd, data preprocessing, github, hugging fa...","[natural language processing, business intelli...","[a, agile, ai, airflow, analytics, apache spar..."
751785,Research Assistant Positions within Policy Lea...,Research Assistant Positions within Policy Lea...,"a, action models, ai, analytics, artificial in...","action models, c plus plus, computer vision, c...","[artificial intelligence, reinforcement learni...","[a, ai, analytics, artificial intelligence, as..."
751786,Senior AI Software Engineer - Indaiatuba/SP,Senior AI Software Engineer - Indaiatuba/SP - ...,"a, ai, airflow, analytics, apache spark, api, ...","api integration, aws, ci, cd, debugging, devop...","[data visualization, machine learning, compute...","[a, ai, airflow, analytics, apache spark, api,..."
751787,[BD] AI Intern,"[BD] AI Intern - Hanoi, Vietnam ai jobs.net Si...","a, agile, ai, airflow, analytics, apache spark...","cloud computing, data quality, decision trees,...","[natural language processing, software enginee...","[a, agile, ai, airflow, analytics, apache spar..."
751788,"Data Science Manager, Supply","Data Science Manager, Supply - San Francisco, ...","a, agile, ai, airflow, analytics, api, artific...","causal inference, data visualization, experime...","[artificial intelligence, business intelligenc...","[a, agile, ai, airflow, analytics, api, artifi..."
751789,Embedded Engineer,"Embedded Engineer - Lviv, Ukraine ai jobs.net ...","a, ai, airflow, apache spark, api, arm, artifi...","arm, c#, c++, csharp, git, i2c, linux, oop, py...","[artificial intelligence, data transformation,...","[a, ai, airflow, apache spark, api, artificial..."
751790,Embedded Software Engineer,"Embedded Software Engineer - Ha Noi, Vietnam a...","a, agile, ai, apache spark, api, aspice, autom...","aspice, autosar, agile, c#, c++, can, doors, d...","[machine learning, data extraction, data colle...","[a, ai, apache spark, api, automation, aws, az..."
